In [1]:
import json
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

e:\Geeta Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
# 2. Initialize persistent ChromaDB - persistent client is saved to disk
chroma_client = chromadb.PersistentClient(path="./gita_chroma_db")

collection = chroma_client.get_or_create_collection(
    name = "bhagavad-gita",
    metadata={"hnsw:space":"cosine"} #using cosine similarity
)

with open('clean_gita_data.json','r',encoding='utf-8') as f:
    gita_data = json.load(f)

documents = []
metadata = []
ids = []

print('preparing data for embedding')

for i in gita_data:
    english_translation = i["english_translation"]
    documents.append(english_translation)

    chapter = i['chapter']
    verse = i['verse']
    speaker = i['speaker']

    metadata.append({"chapter":chapter,"verse":verse,"speaker":speaker})

    unique_id = f"ch{chapter}_v{verse}"
    ids.append(unique_id)

# add to chromadb in batches
print("Generating Embeddings and saving to chromadb")
embedded_docs = embeddings.embed_documents(documents)

collection.add(
    embeddings=embedded_docs,
    documents=documents,
    metadatas=metadata,
    ids=ids
)

print(f"Successfully embedded and stored {collection.count()} verses.")

e:\Geeta Project\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arjun\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10473.79it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key

preparing data for embedding
Generating Embeddings and saving to chromadb
Successfully embedded and stored 701 verses.
